# Forward-Risk-Manager End-to-End Runbook

This single notebook runs the repo workflow from start to finish:
- optional environment install
- graph build (with optional macro + advanced graph controls)
- strict two-stage training (encoder then critic)
- benchmark + sanity checks (walk-forward + slippage-aware economics)
- hyperparameter sweep + summary + plots
- optional Bayesian sweep tuning (Optuna wrapper)
- scenario book + stress test + hallucination calibration
- scenario/feature/embedding diagnostics
- optional goodness backtest
- final artifact inventory

Current implementation status (checked in-repo):
- implemented: critic-aware FF benchmark/econ eval, walk-forward std/min aggregation, stability-aware sweep ranking, regime confidence gating, lead-lag/static graph overlays, leakage-safe graph lags
- partial: adversarial negatives (`edge_attack`) perturb hub features/time order, but edge-drop/sign-flip graph attacks are still pending
- pending in `scripts/train_ff_gnn.py`: critic ensembles/sequence critic integration and portfolio-head utility objective

All outputs are redirected to an isolated run folder under `runs/experiments/`.



In [1]:
import csv
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from collections import deque
from pathlib import Path

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

import pandas as pd
import torch


def q(value) -> str:
    return shlex.quote(str(value))


def run(cmd: str, allow_fail: bool = False, tail_lines: int = 200) -> bool:
    print("\n" + "=" * 110)
    print(cmd)
    print("=" * 110)
    t0 = time.time()

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

    logs_dir = Path("runs/experiments/_e2e_logs")
    logs_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", cmd).strip("_")[:120] or "command"
    log_path = logs_dir / f"{int(time.time())}_{safe_name}.log"

    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None

    tail = deque(maxlen=max(40, int(tail_lines)))
    with log_path.open("w", encoding="utf-8") as lf:
        for line in proc.stdout:
            print(line, end="")
            lf.write(line)
            tail.append(line.rstrip("\n"))
    rc = proc.wait()

    elapsed = time.time() - t0
    print(f"\ncompleted in {elapsed:.2f}s | log: {log_path}")
    if rc != 0:
        if tail:
            print("---- command output tail ----")
            for ln in tail:
                print(ln)
            print("---- end tail ----")
        msg = f"command failed ({rc}): {cmd}"
        if allow_fail:
            print("WARNING:", msg)
            return False
        raise RuntimeError(msg)
    return True


def _toml_value_literal(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return repr(float(value))
    if isinstance(value, list):
        return "[" + ", ".join(_toml_value_literal(v) for v in value) + "]"
    return json.dumps(str(value))


def write_runtime_config(base_config_path: str, runtime_config_path: str, section_overrides: dict) -> str:
    src = Path(base_config_path)
    dst = Path(runtime_config_path)
    lines = src.read_text().splitlines()

    section_re = re.compile(r"^\s*\[([^\]]+)\]\s*$")
    key_res = {
        sec: {k: re.compile(rf"^\s*{re.escape(k)}\s*=") for k in kv}
        for sec, kv in section_overrides.items()
    }
    replaced = {(sec, key): False for sec, kv in section_overrides.items() for key in kv}
    seen_sections = set()

    out = []
    current_section = None

    def flush_missing(section_name):
        if section_name not in section_overrides:
            return
        for key, value in section_overrides[section_name].items():
            if not replaced[(section_name, key)]:
                out.append(f"{key} = {_toml_value_literal(value)}")
                replaced[(section_name, key)] = True

    for line in lines:
        m = section_re.match(line)
        if m:
            flush_missing(current_section)
            current_section = m.group(1).strip()
            seen_sections.add(current_section)
            out.append(line)
            continue

        updated = False
        if current_section in section_overrides:
            for key, pattern in key_res[current_section].items():
                if pattern.match(line):
                    out.append(f"{key} = {_toml_value_literal(section_overrides[current_section][key])}")
                    replaced[(current_section, key)] = True
                    updated = True
                    break

        if not updated:
            out.append(line)

    flush_missing(current_section)

    for section_name, overrides in section_overrides.items():
        if section_name in seen_sections:
            continue
        if out and out[-1].strip() != "":
            out.append("")
        out.append(f"[{section_name}]")
        for key, value in overrides.items():
            out.append(f"{key} = {_toml_value_literal(value)}")

    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(out) + "\n")
    return str(dst)


In [ ]:
# Controls
RUN_PROFILE = "full"  # "full" or "fast"
ECON_PROFILE = "aggressive"  # "aggressive" or "baseline"
INSTALL_DEPS = ("google.colab" in sys.modules)
RUN_SWEEP_PROMOTION = True
RUN_OPTIONAL_BACKTEST = True
RUN_OPTUNA = False
RUN_EXTRA_DIAGNOSTICS = True
RUN_SCENARIO_PREFLIGHT = True
SCENARIO_PREFLIGHT_NUM_SCENARIOS = 1
FORCE_REBUILD_GRAPHS = False
AUTO_PREP_DATA = True
RETRY_STAGE2_NO_COMPILE_ON_FAIL = True

OPTUNA_TRIALS = 8 if RUN_PROFILE == "full" else 3
ATTR_NUM_GRAPHS = 256 if RUN_PROFILE == "full" else 64
EMBED_METHOD = "pca"  # "umap" | "tsne" | "pca" | "svd"

# Optional explicit overrides. Leave empty to auto-detect.
DATA_PRICES_PATH = ""
DATA_CONSTITUENTS_PATH = ""
DATA_FUNDAMENTALS_PATH = ""
DATA_MACRO_PATH = ""

# Graph/model/econ knobs to keep the runbook aligned with current defaults.
GRAPH_CORR_METHOD = "pearson"  # or "partial"
GRAPH_EDGE_SELECT_MODE = "top_k"  # or "significance"
GRAPH_CROSS_SECTIONAL_NORM = False
GRAPH_EDGE_NODE_WEIGHTING = "none"  # "none" | "volume" | "market_cap" | "volume_market_cap"
GRAPH_MEMBERSHIP_FILL = "ffill"  # "none" | "ffill" (as-of carry-forward; no lookahead)
GRAPH_MEMBERSHIP_MAX_GAP_DAYS = 63
TRAIN_ENCODER_CONV_TYPE = "gcn"  # "gcn" | "sage" | "gat"
TRAIN_ENCODER_GAT_HEADS = 2
TRAIN_FF_MARGIN = 0.0
TRAIN_RISK_LOSS_WEIGHT = 0.005
TRAIN_CRITIC_ENSEMBLE_SIZE = 3
TRAIN_SEQUENCE_CRITIC_ENABLED = True
TRAIN_SEQUENCE_CRITIC_WEIGHT = 0.15
TRAIN_RESIDUAL_EDGE_WEIGHT_ENABLED = True
TRAIN_RESIDUAL_EDGE_MAX_DELTA = 0.2
TRAIN_PORTFOLIO_HEAD_ENABLED = True
TRAIN_PORTFOLIO_LOSS_WEIGHT = 0.001
TRAIN_NEG_MODE = "mix"
GRAPH_MACRO_LAG_DAYS = 1

# Forward-Forward signal quality + speed controls.
TRAIN_FF_NEG_MIX = ["time_flip", "sector_swap", "factor_hard", "noise"]
TRAIN_FF_NEG_MIX_WEIGHTS = [0.35, 0.25, 0.25, 0.15]
TRAIN_FF_CURRICULUM_EPOCHS = [0.33, 0.34, 0.33]
TRAIN_FF_RANK_AUX_WEIGHT = 0.02
TRAIN_FF_RANK_USE_PORTFOLIO_TARGETS = True
TRAIN_FF_HALL_EVERY_N_BATCHES = 4
TRAIN_FF_HALL_WARMUP_EPOCHS = 3
TRAIN_FF_HALL_STEPS = 3
TRAIN_FF_CONCAT_POSNEG = True
TRAIN_FF_LAYER_CACHE = True
TRAIN_FF_DUAL_NEG_EVERY_N_BATCHES = 4
TRAIN_FF_ECON_EVAL_EVERY = 10
TRAIN_TORCH_COMPILE = True
TRAIN_TORCH_COMPILE_MODE = "reduce-overhead"

E2E_EXPECTED_GPU_PROFILE = "a100"
E2E_ENFORCE_EXPECTED_GPU = False
E2E_TRAIN_AMP_DTYPE = "bfloat16"
E2E_BACKPROP_AMP_DTYPE = "bfloat16"
TRAIN_AUTO_TUNE_MAX_BATCH = 64

BENCH_TIMING_WARMUP_EPOCHS = 1
BENCH_REQUIRE_BASELINE_MATCH = False
BENCH_BASELINE_REF_CSV = ""
BENCH_TORCH_COMPILE = False
BENCH_TORCH_COMPILE_MODE = "max-autotune-no-cudagraphs"
BENCH_RETRY_SAFE_ON_ERROR = True
BENCH_ENABLE_RISK_HEAD = False
BENCH_ENABLE_PORTFOLIO_HEAD = False
BENCH_BACKPROP_CONCAT_POSNEG = True
BENCH_ECON_REGIME_THRESHOLDING_ENABLED = True
BENCH_ECON_REGIME_THRESHOLD_WINDOW = 126
BENCH_ECON_REGIME_THRESHOLD_QUANTILE = 0.55
BENCH_ECON_REGIME_VOL_WINDOW = 21
BENCH_ECON_REGIME_LOW_QUANTILE = 0.30
BENCH_ECON_REGIME_HIGH_QUANTILE = 0.70

ECON_TURNOVER_COST_BPS = 1.0
ECON_SLIPPAGE_BPS = 0.5
ECON_SLIPPAGE_VOL_SCALE = 0.25
ECON_SLIPPAGE_VOL_LOOKBACK = 21
ECON_REGIME_GATE_ENABLED = True
ECON_REGIME_GATE_WINDOW = 63
ECON_REGIME_CONFIDENCE_TEMP = 1.0
ECON_REGIME_NEUTRAL_EXPOSURE = 0.0
ECON_REGIME_MIN_CONFIDENCE = 0.1
ECON_REGIME_UNCERTAINTY_SCALE = 0.25
ECON_REGIME_RISK_SCALE = 0.25

SWEEP_RANK_MODE = "finance_first"
SWEEP_STABILITY_PENALTY = True
SWEEP_STABILITY_LAMBDA = 0.5
SWEEP_SHARPE_MIN_FLOOR = 0.05
SWEEP_RANK_GATE_PENALTY = 1_000_000.0
SWEEP_NEG_MODE = "shuffle+noise"
SWEEP_EVAL_NEG_MODES = ["time_flip", "block_bootstrap", "cross_asset_mix", "phase_randomize"]
SWEEP_EVAL_NEG_TOP_K = 8 if RUN_PROFILE == "full" else 3
SWEEP_EVAL_NEG_PRE_RANK_MODE = "objective"
SWEEP_EVAL_NEG_AGGREGATE = "mean"
SWEEP_EVAL_NEG_INCLUDE_BASE = True
SWEEP_MODES = ["ff_layerwise", "ff_e2e"]
SWEEP_MAX_RUNS = 24 if RUN_PROFILE == "full" else 8
SWEEP_ECON_TOP_K = 8 if RUN_PROFILE == "full" else 3
SWEEP_SUCCESSIVE_HALVING_ENABLED = True
SWEEP_SUCCESSIVE_HALVING_STAGE_FRACS = [0.34, 1.0]
SWEEP_SUCCESSIVE_HALVING_KEEP_RATIO = 0.5
SWEEP_SUCCESSIVE_HALVING_MIN_KEEP = 6
SWEEP_WALK_FORWARD_MAX_FOLDS_CAP = 3
SWEEP_EARLY_STOP_ENABLED = True
SWEEP_EARLY_STOP_MIN_EPOCHS = 2
SWEEP_EARLY_STOP_PATIENCE = 2
SWEEP_EARLY_STOP_MIN_DELTA = 0.001
SWEEP_FINANCE_SEP_GATE_FLOOR = 0.01
SWEEP_FINANCE_AUROC_GATE_FLOOR = 0.52
SWEEP_ECON_WEIGHT = 0.62
SWEEP_SEP_WEIGHT = 0.28
SWEEP_SPEED_WEIGHT = 0.10
SWEEP_GOODNESS_TARGETS = [2.0, 2.3]
SWEEP_HALL_STEPS = [2, 3]
SWEEP_HALL_LRS = [0.025, 0.035]
SWEEP_HALL_NODE_FRACTIONS = [0.25, 0.35]
SWEEP_NEG_MIX_END = [0.6, 0.75]

assert RUN_PROFILE in {"full", "fast"}
assert ECON_PROFILE in {"aggressive", "baseline"}

if ECON_PROFILE == "baseline":
    SWEEP_SHARPE_MIN_FLOOR = 0.0
    SWEEP_FINANCE_SEP_GATE_FLOOR = 0.002
    SWEEP_FINANCE_AUROC_GATE_FLOOR = ""
    SWEEP_ECON_WEIGHT = 0.45
    SWEEP_SEP_WEIGHT = 0.35
    SWEEP_SPEED_WEIGHT = 0.20
    SWEEP_GOODNESS_TARGETS = [1.5, 2.0]
    SWEEP_HALL_STEPS = [1, 2]
    SWEEP_HALL_LRS = [0.02, 0.03]
    SWEEP_HALL_NODE_FRACTIONS = [0.2]
    SWEEP_NEG_MIX_END = [0.5, 0.7]

IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)
print("INSTALL_DEPS:", INSTALL_DEPS)

# Colab runtime and GPU sanity checks (A100 stage)
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    print("python:", sys.version)
    print("python_executable:", sys.executable)
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    print("cuda_version:", torch.version.cuda)

    try:
        print(subprocess.check_output(["nvidia-smi", "-L"], text=True))
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. In Colab, switch Runtime -> GPU and rerun.")

    gpu_name = torch.cuda.get_device_name(0)
    gpu_upper = gpu_name.upper()
    gpu_profile = "h100" if "H100" in gpu_upper else ("a100" if "A100" in gpu_upper else ("t4" if "T4" in gpu_upper else "generic"))
    print("gpu:", gpu_name, "| profile:", gpu_profile)

    if gpu_profile != E2E_EXPECTED_GPU_PROFILE:
        msg = (
            f"end_to_end_repo_runbook expects GPU profile '{E2E_EXPECTED_GPU_PROFILE}' "
            f"but found '{gpu_profile}' ({gpu_name})."
        )
        if E2E_ENFORCE_EXPECTED_GPU:
            raise RuntimeError(msg)
        print("WARNING:", msg)

    # Throughput-friendly defaults for CUDA/A100.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("FRM_REPO_DIR", "").strip()
    if env_repo and Path(env_repo).exists() and (Path(env_repo) / "configs/default.toml").exists():
        return Path(env_repo)

    candidates = [
        Path.cwd(),
        Path("/content/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/forward-risk-manager"),
    ]

    if IN_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            candidates.extend(
                sorted(
                    p for p in drive_root.glob("*Forward*Risk*Manager*")
                    if p.is_dir()
                )
            )

    for p in candidates:
        if (p / "configs/default.toml").exists():
            return p

    raise FileNotFoundError(
        "Could not find repo root containing configs/default.toml. "
        "Set FRM_REPO_DIR or update candidate paths in this cell."
    )


REPO_DIR = _resolve_repo_dir()
os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("cwd:", Path.cwd())

if Path(".git").exists():
    status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
    if status.returncode == 0:
        out = status.stdout.strip()
        print(out[:4000] if out else "Git repo detected. Working tree clean.")
    else:
        msg = (status.stderr or status.stdout or "").strip()
        print(f"`git status --short` failed with exit code {status.returncode}.")
        if msg:
            print(msg[:4000])
        if "dubious ownership" in msg.lower():
            print('Fix: run `!git config --global --add safe.directory "$PWD"` and retry.')

venv_py = Path(".venv/bin/python")
PYTHON_EXE = venv_py if venv_py.exists() else Path(sys.executable)
print("python exe for commands:", PYTHON_EXE)

BASE_CONFIG = Path("configs/default.toml")
assert BASE_CONFIG.exists(), f"Missing config: {BASE_CONFIG}"

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path("runs/experiments") / f"e2e_runbook_{RUN_ID}"
METRICS_DIR = RUN_ROOT / "metrics"
PLOTS_DIR = RUN_ROOT / "plots"
LOGS_DIR = RUN_ROOT / "logs"
MODELS_DIR = RUN_ROOT / "models"
DIAG_DIR = RUN_ROOT / "diagnostics"
DATA_DIR = RUN_ROOT / "data"

for p in [METRICS_DIR, PLOTS_DIR, LOGS_DIR, MODELS_DIR, DIAG_DIR, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

runtime_config = RUN_ROOT / "runtime_config.toml"

graphs_pt = DATA_DIR / "graphs.pt"
model_ckpt = MODELS_DIR / "ff_model.pt"
encoder_ckpt = MODELS_DIR / "encoder.pt"
critic_ckpt = MODELS_DIR / "critic.pt"

benchmark_csv = METRICS_DIR / "benchmark.csv"
benchmark_baseline_csv = METRICS_DIR / "benchmark_baseline.csv"
benchmark_walk_forward_csv = METRICS_DIR / "benchmark_walk_forward_folds.csv"
benchmark_plot = PLOTS_DIR / "benchmark_speed_sep.png"
benchmark_bar = PLOTS_DIR / "benchmark_bar.png"

sweep_csv = METRICS_DIR / "ff_sweep.csv"
sweep_summary = LOGS_DIR / "ff_sweep_summary.txt"
sweep_plot = PLOTS_DIR / "ff_sweep_tradeoff.png"
sweep_pareto = PLOTS_DIR / "ff_sweep_pareto.png"
optuna_json = METRICS_DIR / "ff_optuna_best.json"

scenario_csv = METRICS_DIR / "scenario_book.csv"
scenario_diag = DIAG_DIR / "scenario_constraint_diagnostics.csv"
scenario_delta_csv = DIAG_DIR / "scenario_delta_report.csv"
stress_csv = METRICS_DIR / "stress_test_report.csv"
stress_plot = PLOTS_DIR / "stress_test_report.png"
calibration_json = DIAG_DIR / "hallucination_calibration.json"
calibration_by_ticker = DIAG_DIR / "hallucination_calibration_by_ticker.csv"
feature_attr_csv = DIAG_DIR / "feature_attribution.csv"
feature_attr_energy_csv = DIAG_DIR / "feature_attribution_graph_energy.csv"
embedding_csv = DIAG_DIR / "embedding_projection.csv"

goodness_csv = DIAG_DIR / "goodness_backtest.csv"
goodness_quantiles = DIAG_DIR / "goodness_quantiles.csv"
goodness_plot = PLOTS_DIR / "goodness_scatter.png"
goodness_events = DIAG_DIR / "goodness_events.csv"
goodness_strategy = DIAG_DIR / "goodness_strategy_metrics.csv"
goodness_timeline = PLOTS_DIR / "goodness_timeline.png"


def _find_tidy_sources() -> tuple[str, Path, Path, Path | None]:
    if DATA_PRICES_PATH or DATA_CONSTITUENTS_PATH:
        prices = Path(DATA_PRICES_PATH) if DATA_PRICES_PATH else None
        consts = Path(DATA_CONSTITUENTS_PATH) if DATA_CONSTITUENTS_PATH else None
        if prices is None or consts is None:
            raise ValueError("Set both DATA_PRICES_PATH and DATA_CONSTITUENTS_PATH, or leave both empty.")
        if not prices.exists() or not consts.exists():
            raise FileNotFoundError(
                f"Explicit data path missing: prices={prices} exists={prices.exists()} | "
                f"constituents={consts} exists={consts.exists()}"
            )
        fund = Path(DATA_FUNDAMENTALS_PATH) if DATA_FUNDAMENTALS_PATH else None
        if fund is not None and not fund.exists():
            print(f"WARNING: DATA_FUNDAMENTALS_PATH missing, ignoring: {fund}")
            fund = None
        return "manual", prices, consts, fund

    candidate_pairs = [
        (
            "processed_long",
            Path("data/processed_long/prices.csv"),
            Path("data/processed_long/constituents.csv"),
            Path("data/processed_long/fundamentals.csv"),
        ),
        (
            "processed",
            Path("data/processed/prices.csv"),
            Path("data/processed/constituents.csv"),
            Path("data/processed/fundamentals.csv"),
        ),
        (
            "raw_merged",
            Path("data/raw_merged/prices.csv"),
            Path("data/raw_merged/constituents.csv"),
            Path("data/raw_merged/fundamentals.csv"),
        ),
    ]

    # Add recursive fallback discovery under data/.
    data_root = Path("data")
    if data_root.exists():
        for price_path in sorted(data_root.rglob("prices.csv")):
            const_path = price_path.with_name("constituents.csv")
            fund_path = price_path.with_name("fundamentals.csv")
            if const_path.exists():
                candidate_pairs.append((f"discovered:{price_path.parent}", price_path, const_path, fund_path))

    for label, prices, consts, fund in candidate_pairs:
        if prices.exists() and consts.exists():
            return label, prices, consts, fund if fund.exists() else None

    # Optional auto-prepare path for year-bucketed raw exports.
    if AUTO_PREP_DATA and Path("data/raw").exists():
        print("No tidy prices/constituents found. Attempting merge_raw_years auto-prepare...")
        run(
            f"{q(PYTHON_EXE)} scripts/merge_raw_years.py --raw-root data/raw --out-dir data/raw_merged",
            allow_fail=False,
        )
        prices = Path("data/raw_merged/prices.csv")
        consts = Path("data/raw_merged/constituents.csv")
        fund = Path("data/raw_merged/fundamentals.csv")
        if prices.exists() and consts.exists():
            return "raw_merged(auto_prepared)", prices, consts, fund if fund.exists() else None

    raise FileNotFoundError(
        "Could not find tidy inputs for graph building (prices.csv + constituents.csv).\n"
        "Provide DATA_PRICES_PATH/DATA_CONSTITUENTS_PATH in this cell, or prepare data via one of:\n"
        "1) python scripts/qc_export_to_tidy.py --prices ... --constituents ... --out-dir data/processed\n"
        "2) python scripts/merge_raw_years.py --raw-root data/raw --out-dir data/raw_merged"
    )


def _resolve_macro_source(prices_path: Path) -> Path | None:
    if DATA_MACRO_PATH:
        p = Path(DATA_MACRO_PATH)
        if p.exists():
            return p
        print(f"WARNING: DATA_MACRO_PATH missing, ignoring: {p}")
        return None

    candidates = [
        prices_path.with_name("macro.csv"),
        Path("data/processed_long/macro.csv"),
        Path("data/processed/macro.csv"),
        Path("data/raw_merged/macro.csv"),
        Path("data/raw_merged/macro_prices.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


data_source_label, build_prices_path, build_constituents_path, build_fundamentals_path = _find_tidy_sources()
build_macro_path = _resolve_macro_source(build_prices_path)

if build_fundamentals_path is None:
    fallback_fund = Path("data/raw_merged/fundamentals.csv")
    if fallback_fund.exists():
        build_fundamentals_path = fallback_fund

print("data source:", data_source_label)
print("build prices:", build_prices_path)
print("build constituents:", build_constituents_path)
print("build fundamentals:", build_fundamentals_path if build_fundamentals_path else "<none>")
print("build macro:", build_macro_path if build_macro_path else "<none>")

overrides = {
    "build_graphs": {
        "prices": str(build_prices_path),
        "constituents": str(build_constituents_path),
        "out": str(graphs_pt),
        "corr_method": GRAPH_CORR_METHOD,
        "edge_select_mode": GRAPH_EDGE_SELECT_MODE,
        "cross_sectional_norm": GRAPH_CROSS_SECTIONAL_NORM,
        "edge_node_weighting": GRAPH_EDGE_NODE_WEIGHTING,
        "membership_fill": GRAPH_MEMBERSHIP_FILL,
        "membership_max_gap_days": GRAPH_MEMBERSHIP_MAX_GAP_DAYS,
        "macro_lag_days": GRAPH_MACRO_LAG_DAYS,
    },
    "train": {
        "graphs": str(graphs_pt),
        "save_model": str(model_ckpt),
        "save_encoder": str(encoder_ckpt),
        "save_critic": str(critic_ckpt),
        "log_csv": str(METRICS_DIR / "ff_train.csv"),
        "plot_path": str(PLOTS_DIR / "ff_train.png"),
        "strict_component_split": True,
        "encoder_conv_type": TRAIN_ENCODER_CONV_TYPE,
        "encoder_gat_heads": TRAIN_ENCODER_GAT_HEADS,
        "ff_margin": TRAIN_FF_MARGIN,
        "ff_margin_weight": 1.0,
        "risk_loss_weight": TRAIN_RISK_LOSS_WEIGHT,
        "critic_ensemble_size": TRAIN_CRITIC_ENSEMBLE_SIZE,
        "sequence_critic_enabled": TRAIN_SEQUENCE_CRITIC_ENABLED,
        "sequence_critic_weight": TRAIN_SEQUENCE_CRITIC_WEIGHT,
        "residual_edge_weight_enabled": TRAIN_RESIDUAL_EDGE_WEIGHT_ENABLED,
        "residual_edge_max_delta": TRAIN_RESIDUAL_EDGE_MAX_DELTA,
        "portfolio_head_enabled": TRAIN_PORTFOLIO_HEAD_ENABLED,
        "portfolio_loss_weight": TRAIN_PORTFOLIO_LOSS_WEIGHT,
        "neg_mode": TRAIN_NEG_MODE,
        "ff_neg_mix": TRAIN_FF_NEG_MIX,
        "ff_neg_mix_weights": TRAIN_FF_NEG_MIX_WEIGHTS,
        "ff_curriculum_epochs": TRAIN_FF_CURRICULUM_EPOCHS,
        "ff_rank_aux_weight": TRAIN_FF_RANK_AUX_WEIGHT,
        "ff_rank_use_portfolio_targets": TRAIN_FF_RANK_USE_PORTFOLIO_TARGETS,
        "ff_hall_every_n_batches": TRAIN_FF_HALL_EVERY_N_BATCHES,
        "ff_hall_warmup_epochs": TRAIN_FF_HALL_WARMUP_EPOCHS,
        "ff_hall_steps": TRAIN_FF_HALL_STEPS,
        "ff_concat_posneg": TRAIN_FF_CONCAT_POSNEG,
        "ff_layer_cache": TRAIN_FF_LAYER_CACHE,
        "ff_dual_neg_every_n_batches": TRAIN_FF_DUAL_NEG_EVERY_N_BATCHES,
        "ff_econ_eval_every": TRAIN_FF_ECON_EVAL_EVERY,
        "torch_compile": TRAIN_TORCH_COMPILE,
        "torch_compile_mode": TRAIN_TORCH_COMPILE_MODE,
        "amp_dtype": E2E_TRAIN_AMP_DTYPE,
        "auto_tune_max_batch": TRAIN_AUTO_TUNE_MAX_BATCH,
        "auto_tune_cache": True,
        "auto_tune_cache_path": "runs/cache/auto_tune_batch.json",
        "hallucinate_adaptive_lr": True,
        "hallucinate_adaptive_lr_patience": 2,
        "hallucinate_adaptive_lr_decay": 0.5,
        "hallucinate_adaptive_lr_min": 1e-4,
        "hallucinate_early_stop_on_target_hit": True,
        "hallucinate_target_hit_patience": 2,
        "hallucinate_moment_mean": 0.002,
        "hallucinate_moment_var": 0.01,
        "hallucinate_moment_skew": 0.002,
        "hallucinate_moment_scope": "returns",
    },
    "benchmark": {
        "out_csv": str(benchmark_csv),
        "walk_forward_out_csv": str(benchmark_walk_forward_csv),
        "plot_path": str(benchmark_plot),
        "bar_plot_path": str(benchmark_bar),
        "split_mode": "walk_forward",
        "walk_forward_train_frac": 0.6,
        "walk_forward_eval_frac": 0.2,
        "walk_forward_step_frac": 0.1,
        "walk_forward_min_train_graphs": 128,
        "walk_forward_min_eval_graphs": 32,
        "neg_mode": SWEEP_NEG_MODE,
        "eval_neg_mode": "auto",
        "eval_neg_modes": SWEEP_EVAL_NEG_MODES,
        "econ_turnover_cost_bps": ECON_TURNOVER_COST_BPS,
        "econ_slippage_bps": ECON_SLIPPAGE_BPS,
        "econ_slippage_vol_scale": ECON_SLIPPAGE_VOL_SCALE,
        "econ_slippage_vol_lookback": ECON_SLIPPAGE_VOL_LOOKBACK,
        "econ_regime_gate_enabled": ECON_REGIME_GATE_ENABLED,
        "econ_regime_gate_window": ECON_REGIME_GATE_WINDOW,
        "econ_regime_confidence_temp": ECON_REGIME_CONFIDENCE_TEMP,
        "econ_regime_neutral_exposure": ECON_REGIME_NEUTRAL_EXPOSURE,
        "econ_regime_min_confidence": ECON_REGIME_MIN_CONFIDENCE,
        "econ_regime_uncertainty_scale": ECON_REGIME_UNCERTAINTY_SCALE,
        "econ_regime_risk_scale": ECON_REGIME_RISK_SCALE,
        "econ_regime_thresholding_enabled": BENCH_ECON_REGIME_THRESHOLDING_ENABLED,
        "econ_regime_threshold_window": BENCH_ECON_REGIME_THRESHOLD_WINDOW,
        "econ_regime_threshold_quantile": BENCH_ECON_REGIME_THRESHOLD_QUANTILE,
        "econ_regime_vol_window": BENCH_ECON_REGIME_VOL_WINDOW,
        "econ_regime_low_quantile": BENCH_ECON_REGIME_LOW_QUANTILE,
        "econ_regime_high_quantile": BENCH_ECON_REGIME_HIGH_QUANTILE,
        "timing_warmup_epochs": BENCH_TIMING_WARMUP_EPOCHS,
        "require_baseline_match": BENCH_REQUIRE_BASELINE_MATCH,
        "baseline_ref_csv": BENCH_BASELINE_REF_CSV,
        "torch_compile": BENCH_TORCH_COMPILE,
        "torch_compile_mode": BENCH_TORCH_COMPILE_MODE,
        "retry_safe_on_error": BENCH_RETRY_SAFE_ON_ERROR,
        "enable_risk_head": BENCH_ENABLE_RISK_HEAD,
        "enable_portfolio_head": BENCH_ENABLE_PORTFOLIO_HEAD,
        "backprop_concat_posneg": BENCH_BACKPROP_CONCAT_POSNEG,
        "amp_dtype": E2E_TRAIN_AMP_DTYPE,
        "backprop_amp_dtype": E2E_BACKPROP_AMP_DTYPE,
        "baseline_out_csv": str(benchmark_baseline_csv),
    },
    "sweep": {
        "out_csv": str(sweep_csv),
        "split_mode": "walk_forward",
        "walk_forward_train_frac": 0.6,
        "walk_forward_eval_frac": 0.2,
        "walk_forward_step_frac": 0.1,
        "walk_forward_min_train_graphs": 128,
        "walk_forward_min_eval_graphs": 32,
        "walk_forward_max_folds_cap": SWEEP_WALK_FORWARD_MAX_FOLDS_CAP,
        "neg_mode": SWEEP_NEG_MODE,
        "eval_neg_mode": "auto",
        "eval_neg_modes": SWEEP_EVAL_NEG_MODES,
        "eval_neg_top_k": SWEEP_EVAL_NEG_TOP_K,
        "eval_neg_pre_rank_mode": SWEEP_EVAL_NEG_PRE_RANK_MODE,
        "eval_neg_aggregate": SWEEP_EVAL_NEG_AGGREGATE,
        "eval_neg_aggregate_include_base": SWEEP_EVAL_NEG_INCLUDE_BASE,
        "successive_halving_enabled": SWEEP_SUCCESSIVE_HALVING_ENABLED,
        "successive_halving_stage_fracs": SWEEP_SUCCESSIVE_HALVING_STAGE_FRACS,
        "successive_halving_keep_ratio": SWEEP_SUCCESSIVE_HALVING_KEEP_RATIO,
        "successive_halving_min_keep": SWEEP_SUCCESSIVE_HALVING_MIN_KEEP,
        "successive_halving_rank_mode": "objective",
        "successive_halving_disable_eval_neg_before_final": True,
        "modes": SWEEP_MODES,
        "max_runs": SWEEP_MAX_RUNS,
        "econ_top_k": SWEEP_ECON_TOP_K,
        "econ_pre_rank_mode": "objective",
        "eval_batch_size": 64,
        "eval_loader_workers": 0,
        "eval_pin_memory": True,
        "early_stop_enabled": SWEEP_EARLY_STOP_ENABLED,
        "early_stop_min_epochs": SWEEP_EARLY_STOP_MIN_EPOCHS,
        "early_stop_patience": SWEEP_EARLY_STOP_PATIENCE,
        "early_stop_min_delta": SWEEP_EARLY_STOP_MIN_DELTA,
        "early_stop_eval_every": 1,
        "early_stop_eval_graphs": 128,
        "early_stop_rank_mode": "objective",
        "top_k": 10,
        "amp_dtype": E2E_TRAIN_AMP_DTYPE,
        "parallel_force_cpu": False,
        "econ_turnover_cost_bps": ECON_TURNOVER_COST_BPS,
        "econ_slippage_bps": ECON_SLIPPAGE_BPS,
        "econ_slippage_vol_scale": ECON_SLIPPAGE_VOL_SCALE,
        "econ_slippage_vol_lookback": ECON_SLIPPAGE_VOL_LOOKBACK,
        "econ_regime_gate_enabled": ECON_REGIME_GATE_ENABLED,
        "econ_regime_gate_window": ECON_REGIME_GATE_WINDOW,
        "econ_regime_confidence_temp": ECON_REGIME_CONFIDENCE_TEMP,
        "econ_regime_neutral_exposure": ECON_REGIME_NEUTRAL_EXPOSURE,
        "econ_regime_min_confidence": ECON_REGIME_MIN_CONFIDENCE,
        "econ_regime_uncertainty_scale": ECON_REGIME_UNCERTAINTY_SCALE,
        "econ_regime_risk_scale": ECON_REGIME_RISK_SCALE,
        "econ_regime_thresholding_enabled": BENCH_ECON_REGIME_THRESHOLDING_ENABLED,
        "econ_regime_threshold_window": BENCH_ECON_REGIME_THRESHOLD_WINDOW,
        "econ_regime_threshold_quantile": BENCH_ECON_REGIME_THRESHOLD_QUANTILE,
        "econ_regime_vol_window": BENCH_ECON_REGIME_VOL_WINDOW,
        "econ_regime_low_quantile": BENCH_ECON_REGIME_LOW_QUANTILE,
        "econ_regime_high_quantile": BENCH_ECON_REGIME_HIGH_QUANTILE,
        "rank_mode": SWEEP_RANK_MODE,
        "stability_penalty": SWEEP_STABILITY_PENALTY,
        "stability_penalty_lambda": SWEEP_STABILITY_LAMBDA,
        "econ_oos_sharpe_uplift_min_floor": SWEEP_SHARPE_MIN_FLOOR,
        "econ_sharpe_uplift_min_floor": SWEEP_SHARPE_MIN_FLOOR,
        "finance_sep_gate_metric": "eval_sep_agg_min",
        "finance_sep_gate_floor": SWEEP_FINANCE_SEP_GATE_FLOOR,
        "finance_auroc_gate_metric": "eval_auroc_agg_min",
        "finance_auroc_gate_floor": SWEEP_FINANCE_AUROC_GATE_FLOOR,
        "rank_gate_penalty": SWEEP_RANK_GATE_PENALTY,
        "composite_sep_metric": "eval_sep_agg_mean",
        "econ_weight": SWEEP_ECON_WEIGHT,
        "sep_weight": SWEEP_SEP_WEIGHT,
        "speed_weight": SWEEP_SPEED_WEIGHT,
        "modes": SWEEP_MODES,
        "goodness_target": SWEEP_GOODNESS_TARGETS,
        "neg_mix_end": SWEEP_NEG_MIX_END,
        "hall_steps": SWEEP_HALL_STEPS,
        "hall_lr": SWEEP_HALL_LRS,
        "hall_node_fraction": SWEEP_HALL_NODE_FRACTIONS,
    },
    "scenario_book": {
        "critic_model": str(critic_ckpt),
        "out": str(scenario_csv),
        "diag_out": str(scenario_diag),
        "target_hit_rate": 0.7,
        "target_tolerance": 0.01,
        "hall_adaptive_lr": True,
        "hall_adaptive_lr_patience": 2,
        "hall_adaptive_lr_decay": 0.5,
        "hall_adaptive_lr_min": 1e-4,
        "early_stop_on_target_hit": True,
        "target_hit_patience": 2,
        "hall_moment_mean": 0.002,
        "hall_moment_var": 0.01,
        "hall_moment_skew": 0.002,
    },
    "encoder": {
        "neg_mode": "self_contrastive",
        "strict_component_split": True,
        "freeze_critic": True,
        "save_encoder": str(encoder_ckpt),
    },
    "critic": {
        "neg_mode": "time_flip+noise",
        "strict_component_split": True,
        "freeze_encoder": True,
        "encoder_checkpoint_in": str(encoder_ckpt),
        "save_critic": str(critic_ckpt),
    },
}

if build_fundamentals_path is not None:
    overrides["build_graphs"]["fundamentals"] = str(build_fundamentals_path)
if build_macro_path is not None:
    overrides["build_graphs"]["macro"] = str(build_macro_path)

if RUN_PROFILE == "fast":
    overrides["train"].update({
        "epochs": 25,
        "batch_size": 16,
    })
    overrides["benchmark"].update({
        "epochs": 3,
        "batch_size": 16,
    })
    overrides["sweep"].update({
        "epochs": 2,
        "batch_size": 16,
        "goodness_temp": [0.2],
        "goodness_target": [1.5],
        "neg_mix_end": [0.5],
        "hall_steps": [1],
        "hall_lr": [0.02],
        "hall_node_fraction": [0.2],
        "max_runs": 4,
        "econ_top_k": 2,
        "eval_neg_top_k": 2,
        "successive_halving_stage_fracs": [0.5, 1.0],
        "successive_halving_min_keep": 2,
        "top_k": 3,
    })
    overrides["scenario_book"].update({
        "num_scenarios": 12,
        "max_adapt_steps": 8,
    })
    overrides["encoder"].update({
        "epochs": 12,
        "batch_size": 16,
    })
    overrides["critic"].update({
        "epochs": 12,
        "batch_size": 16,
    })

write_runtime_config(str(BASE_CONFIG), str(runtime_config), overrides)
print("runtime config:", runtime_config)
print("run root:", RUN_ROOT)


In [3]:
import importlib.util

required_modules = [
    "torch",
    "torch_geometric",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "joblib",
]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

need_install = bool(INSTALL_DEPS) or bool(missing)
if need_install:
    if missing and not INSTALL_DEPS:
        print("Missing modules detected:", missing)
        print("Auto-installing requirements to satisfy missing dependencies.")
    run(f"{q(PYTHON_EXE)} -m pip install --upgrade pip")
    run(f"{q(PYTHON_EXE)} -m pip install -r requirements.txt")
    run(f"{q(PYTHON_EXE)} -m pip install -e .")
else:
    print("Dependencies already present. Skipping install.")

# Verify critical import before starting expensive pipeline steps.
import torch_geometric  # noqa: F401
print("torch_geometric import OK")




/usr/bin/python3 -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

completed in 4.10s | log: runs/experiments/_e2e_logs/1771818120_usr_bin_python3_-m_pip_install_--upgrade_pip.log

/usr/bin/python3 -m pip install -r requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.5 MB/s  0:00:00

completed in 2.81s | log: runs/experiments/_e2e_logs/1771818124_usr_bin_python3_-m_pip_install_-r_requirements.txt.log

/usr/bin/python3 -m pip install -e .
Obtaining file:///content/drive/MyDrive/Forward-Risk-Manager
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to bui

## Step 1: Build Graphs

In [4]:
if FORCE_REBUILD_GRAPHS or not graphs_pt.exists():
    run(f"{q(PYTHON_EXE)} scripts/build_graphs.py --config {q(runtime_config)}")
else:
    print("Graphs already exist, skipping rebuild:", graphs_pt)

assert graphs_pt.exists(), f"Missing graphs payload: {graphs_pt}"

try:
    payload = torch.load(graphs_pt, map_location="cpu", weights_only=False)
except TypeError:
    payload = torch.load(graphs_pt, map_location="cpu")

graphs = payload["graphs"] if isinstance(payload, dict) else payload
dates = payload.get("dates", []) if isinstance(payload, dict) else []
tickers = payload.get("tickers", []) if isinstance(payload, dict) else []
print("num_graphs:", len(graphs))
print("num_dates:", len(dates))
print("has_tickers:", bool(tickers))
if graphs:
    g0 = graphs[0]
    print("sample graph nodes:", int(g0.num_nodes), "features:", int(g0.x.shape[1]))


/usr/bin/python3 scripts/build_graphs.py --config runs/experiments/e2e_runbook_20260223_034158/runtime_config.toml
Membership ffill: source_dates=2659 filled_dates=1486 gap_dropped=134 max_gap_days=63
Macro features: source=file cols=30 rows=6564

Building graphs: 100%|██████████| 3550/3550 [03:54<00:00, 15.16win/s]
Wrote runs/experiments/e2e_runbook_20260223_034158/data/graphs.pt with 3550 graphs
Date range: 2011-06-01 -> 2026-02-06 | windows: 6527 | built: 3550 | skipped_lag=0, skipped: members=2977, cols=0, min_nodes=0, no_edges=0

completed in 316.63s | log: runs/experiments/_e2e_logs/1771818143_usr_bin_python3_scripts_build_graphs.py_--config_runs_experiments_e2e_runbook_20260223_034158_runtime_config.toml.log
num_graphs: 3550
num_dates: 3550
has_tickers: True
sample graph nodes: 319 features: 55


## Step 2: Two-Stage Training (Encoder + Critic)

In [5]:
two_stage_cmd = (
    f"{q(PYTHON_EXE)} scripts/train_two_stage.py "
    f"--config {q(runtime_config)} "
    f"--encoder-out {q(encoder_ckpt)} "
    f"--critic-out {q(critic_ckpt)} "
    f"--critic-neg-mode time_flip+noise"
)

if RETRY_STAGE2_NO_COMPILE_ON_FAIL:
    first_ok = run(two_stage_cmd, allow_fail=True)
    if not first_ok:
        print("Step 2 retry: rerunning with --stage2-no-torch-compile")
        run(two_stage_cmd + " --stage2-no-torch-compile")
else:
    run(two_stage_cmd)

assert encoder_ckpt.exists(), f"Missing encoder checkpoint: {encoder_ckpt}"
assert critic_ckpt.exists(), f"Missing critic checkpoint: {critic_ckpt}"

# Keep save_model aligned for scripts that load train.save_model (e.g., goodness_backtest).
if not model_ckpt.exists() and encoder_ckpt.exists():
    shutil.copy2(encoder_ckpt, model_ckpt)

print("encoder bytes:", encoder_ckpt.stat().st_size)
print("critic bytes:", critic_ckpt.stat().st_size)
print("model bytes:", model_ckpt.stat().st_size if model_ckpt.exists() else 0)



/usr/bin/python3 scripts/train_two_stage.py --config runs/experiments/e2e_runbook_20260223_034158/runtime_config.toml --encoder-out runs/experiments/e2e_runbook_20260223_034158/models/encoder.pt --critic-out runs/experiments/e2e_runbook_20260223_034158/models/critic.pt --critic-neg-mode time_flip+noise
adaptive_goodness_target disabled for self_contrastive mode.
device request: auto
device: cuda
torch: 2.10.0+cu128
cuda_available: True
cuda_version: 12.8
mps_built: False
mps_available: False
cuda_device_name: NVIDIA A100-SXM4-40GB
neg_mode: self_contrastive | batch_size: 32 | loader_workers: 0
ff_mode: layerwise=False, blockwise=False, block_size=2, multiscale=True
energy_penalty: 0.0001 (mode=last)
risk_head: ticker=AUTO horizons=[21] weight=0.005 type=huber std=True max_abs_logret=0.5
portfolio_head: ticker=AUTO horizon=21 weight=0.001 type=sharpe std=True max_abs_logret=0.5
ff_rank_aux: weight=0.02 portfolio_targets=True
critic_arch: ensemble=3, seq_enabled=True, seq_weight=0.15
re

## Step 3: Benchmark + Sanity Checks

In [6]:
print("Benchmark safety/perf policy:")
print(
    "  torch_compile=", BENCH_TORCH_COMPILE,
    "mode=", BENCH_TORCH_COMPILE_MODE,
    "retry_safe_on_error=", BENCH_RETRY_SAFE_ON_ERROR,
    "enable_risk_head=", BENCH_ENABLE_RISK_HEAD,
    "enable_portfolio_head=", BENCH_ENABLE_PORTFOLIO_HEAD,
    "backprop_concat_posneg=", BENCH_BACKPROP_CONCAT_POSNEG,
)
run(f"{q(PYTHON_EXE)} scripts/benchmark_training.py --config {q(runtime_config)}")
sanity_ok = run(
    f"{q(PYTHON_EXE)} scripts/sanity_checks.py --benchmark-csv {q(benchmark_csv)}",
    allow_fail=True,
)
print("sanity_checks_passed:", sanity_ok)

assert benchmark_csv.exists(), f"Missing benchmark CSV: {benchmark_csv}"
benchmark_df = pd.read_csv(benchmark_csv)

required_cols = ["mode", "eval_objective", "primary_eval_metric"]
missing = [c for c in required_cols if c not in benchmark_df.columns]
assert not missing, f"Benchmark missing columns: {missing}"

if "objective_track" not in benchmark_df.columns:
    def _objective_track_fallback(obj):
        o = str(obj).strip().lower()
        if o == "self_contrastive":
            return "encoder"
        if o in {"ff", "forward_forward", "forward-forward"} or o.startswith("ff_"):
            return "critic"
        if o in {"bce", "backprop"}:
            return "classifier"
        return "unknown"

    benchmark_df["objective_track"] = benchmark_df["eval_objective"].apply(_objective_track_fallback)

if "primary_eval_metric_name" not in benchmark_df.columns:
    benchmark_df["primary_eval_metric_name"] = benchmark_df["eval_objective"].apply(
        lambda o: "eval_sc_gap" if str(o).strip().lower() == "self_contrastive"
        else ("eval_auroc" if str(o).strip().lower() in {"bce", "backprop"} else "eval_sep")
    )

print("benchmark rows:", len(benchmark_df))
print("objective_track counts:")
print(benchmark_df["objective_track"].astype(str).value_counts(dropna=False))

if benchmark_walk_forward_csv.exists():
    bwf = pd.read_csv(benchmark_walk_forward_csv)
    print("walk-forward fold rows:", len(bwf))

if benchmark_baseline_csv.exists():
    bbase = pd.read_csv(benchmark_baseline_csv)
    print("baseline rows:", len(bbase))
    bcols = [c for c in ["mode", "avg_epoch_s", "graphs_per_s", "baseline_seed", "baseline_split_mode", "baseline_device"] if c in bbase.columns]
    if bcols:
        print(bbase[bcols].to_string(index=False))

view_cols = [c for c in [
    "mode", "row_type", "split_mode_effective", "walk_forward_num_folds", "eval_objective", "objective_track",
    "eval_sep", "eval_sc_gap", "eval_auroc", "graphs_per_s",
    "econ_ann_return_uplift", "econ_sharpe_uplift", "econ_sharpe_uplift_std", "econ_sharpe_uplift_min",
    "econ_sortino_uplift", "econ_calmar_uplift",
    "econ_regime_confidence_mean", "econ_regime_exposure_mean",
    "econ_regime_thresholding_enabled", "econ_regime_threshold_window", "econ_regime_threshold_quantile",
    "econ_regime_low_count", "econ_regime_mid_count", "econ_regime_high_count",
    "time_neg_gen_s", "time_hallucinate_s", "time_forward_pos_s", "time_forward_neg_s",
    "time_loss_terms_s", "time_optimizer_s", "time_econ_eval_s",
    "baseline_seed", "baseline_device", "baseline_graphs_total", "baseline_batch_size",
    "econ_slippage_bps", "econ_slippage_vol_scale",
] if c in benchmark_df.columns]
print(benchmark_df[view_cols].head(20).to_string(index=False))


Benchmark safety/perf policy:
  torch_compile= False mode= max-autotune-no-cudagraphs retry_safe_on_error= True enable_risk_head= False enable_portfolio_head= False backprop_concat_posneg= True

/usr/bin/python3 scripts/benchmark_training.py --config runs/experiments/e2e_runbook_20260223_034158/runtime_config.toml
econ ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439
walk-forward splits=3 (train_frac=0.6, eval_frac=0.2, step_frac=0.1)
benchmark resiliency: retry_safe_on_error=True, continue_on_mode_error=True, torch_compile=False (mode=max-autotune-no-cudagraphs)

Benchmark: 100%|██████████| 5/5 [00:22<00:00,  4.45s/epoch]
calibrated goodness_target=9.3184 (train-cal acc=0.6763)

Benchmark: 100%|██████████| 5/5 [00:25<00:00,  5.07s/epoch]
calibrated goodness_target=9.4354 (train-cal acc=0.7157)

Benchmark: 100%|██████████| 5/5 [00:28<00:00,  5.76s/epoch]
calibrated goodness_target=8.6320 (train-cal acc=0.7254)

Benchmark: 100%|██████████| 5/5 [00:08<00:00,  1.69s/ep

## Step 4: Sweep + Optional Bayesian Tuning + Summary + Tradeoff Plots



In [7]:
run(f"{q(PYTHON_EXE)} scripts/ff_sweep.py --config {q(runtime_config)}")

if RUN_OPTUNA:
    run(
        f"{q(PYTHON_EXE)} scripts/ff_optuna.py "
        f"--config {q(runtime_config)} "
        f"--section sweep "
        f"--trials {int(OPTUNA_TRIALS)} "
        f"--out-json {q(optuna_json)}"
    )
else:
    print("Skipping ff_optuna (RUN_OPTUNA=False)")

run(f"{q(PYTHON_EXE)} scripts/ff_sweep_summary.py --csv {q(sweep_csv)} --out {q(sweep_summary)}")
run(f"{q(PYTHON_EXE)} scripts/plot_ff_sweep.py --csv {q(sweep_csv)} --out {q(sweep_plot)} --pareto-out {q(sweep_pareto)}")

if RUN_SWEEP_PROMOTION:
    run(f"{q(PYTHON_EXE)} scripts/promote_sweep_best.py --config {q(runtime_config)} --csv {q(sweep_csv)} --rank-by rank_value --top-k 5")
else:
    print("Skipping promote_sweep_best (RUN_SWEEP_PROMOTION=False)")

assert sweep_csv.exists(), f"Missing sweep CSV: {sweep_csv}"
sweep_df = pd.read_csv(sweep_csv)

if "objective_track" not in sweep_df.columns:
    def _objective_track_fallback(obj):
        o = str(obj).strip().lower()
        if o == "self_contrastive":
            return "encoder"
        if o in {"ff", "forward_forward", "forward-forward"} or o.startswith("ff_"):
            return "critic"
        if o in {"bce", "backprop"}:
            return "classifier"
        return "unknown"

    sweep_df["objective_track"] = sweep_df["eval_objective"].apply(_objective_track_fallback)

if "primary_eval_metric_name" not in sweep_df.columns:
    sweep_df["primary_eval_metric_name"] = sweep_df["eval_objective"].apply(
        lambda o: "eval_sc_gap" if str(o).strip().lower() == "self_contrastive"
        else ("eval_auroc" if str(o).strip().lower() in {"bce", "backprop"} else "eval_sep")
    )

print("sweep rows:", len(sweep_df))
print("objective_track counts:")
print(sweep_df["objective_track"].astype(str).value_counts(dropna=False))

sort_col = "rank_value" if "rank_value" in sweep_df.columns else ("score" if "score" in sweep_df.columns else "primary_eval_metric")
preview_cols = [c for c in [
    "mode", "split_mode_effective", "walk_forward_num_folds", "eval_objective", "objective_track",
    "rank_base_metric", "rank_base_value", "rank_metric", "rank_value", "rank_gate_failed", "score",
    "econ_weight", "sep_weight", "speed_weight",
    "primary_eval_metric_name", "primary_eval_metric", "graphs_per_s",
    "econ_ann_return_uplift", "econ_sharpe_uplift", "econ_sharpe_uplift_std", "econ_sharpe_uplift_min",
    "econ_sharpe_uplift_stability_adj", "econ_sortino_uplift", "econ_calmar_uplift",
] if c in sweep_df.columns]
print(sweep_df.sort_values(sort_col, ascending=False).head(10)[preview_cols].to_string(index=False))

if optuna_json.exists():
    with Path(optuna_json).open() as f:
        optuna_payload = json.load(f)
    print("optuna summary:", {
        "search_mode": optuna_payload.get("search_mode"),
        "trials": optuna_payload.get("trials"),
        "best_score": optuna_payload.get("best_score"),
        "best_params": optuna_payload.get("best_params"),
    })



/usr/bin/python3 scripts/ff_sweep.py --config runs/experiments/e2e_runbook_20260223_034158/runtime_config.toml
econ ticker: requested=AUTO effective=BWC source=auto_max_rows rows=10439

Sweep: 100%|██████████| 256/256 [3:46:04<00:00, 52.99s/trial]
Wrote runs/experiments/e2e_runbook_20260223_034158/metrics/ff_sweep.csv
Best by rank_value (econ_sharpe_uplift_stability_adj): {'avg_epoch_s': 2.064775659499901, 'avg_epoch_s_std': 0.30178602991104303, 'avg_epoch_s_min': 1.756153419999464, 'avg_epoch_s_max': 2.3592261485000563, 'econ_ann_return_uplift': -0.11145901852846658, 'econ_ann_return_uplift_std': 0.03609521518746957, 'econ_ann_return_uplift_min': -0.13241985075923202, 'econ_ann_return_uplift_max': -0.06978008973098615, 'econ_avg_cost_bps_applied': 0.3296768245171062, 'econ_avg_cost_bps_applied_std': 0.09143574295111591, 'econ_avg_cost_bps_applied_min': 0.26579120118520455, 'econ_avg_cost_bps_applied_max': 0.43441706359394383, 'econ_bh_ann_return': 0.23206487058306827, 'econ_bh_ann_re

## Step 5: Scenario Book + Stress + Calibration + Diagnostics



In [ ]:
scenario_preflight_ok = True
scenario_stage_ran = False
scenario_preflight_csv = scenario_csv.with_name("scenario_book_preflight.csv")
scenario_preflight_diag = scenario_diag.with_name("scenario_book_preflight_diag.csv")

scenario_base_cmd = (
    f"{q(PYTHON_EXE)} scripts/scenario_book.py "
    f"--config {q(runtime_config)} "
    f"--critic-model {q(critic_ckpt)} "
)

if RUN_SCENARIO_PREFLIGHT:
    preflight_cmd = (
        scenario_base_cmd
        + f"--num-scenarios {int(SCENARIO_PREFLIGHT_NUM_SCENARIOS)} "
        + f"--out {q(scenario_preflight_csv)} "
        + f"--diag-out {q(scenario_preflight_diag)}"
    )
    scenario_preflight_ok = run(preflight_cmd, allow_fail=True)
    if not scenario_preflight_ok:
        print("WARNING: scenario preflight failed. Skipping scenario-dependent downstream steps.")
        print("preflight csv:", scenario_preflight_csv)
        print("preflight diag:", scenario_preflight_diag)

if scenario_preflight_ok:
    run(
        scenario_base_cmd
        + f"--out {q(scenario_csv)} "
        + f"--diag-out {q(scenario_diag)}"
    )
    scenario_stage_ran = True
else:
    print("Skipping full scenario_book run because preflight failed.")

if scenario_stage_ran:
    run(
        f"{q(PYTHON_EXE)} scripts/stress_test_report.py "
        f"--csv {q(scenario_csv)} "
        f"--out-csv {q(stress_csv)} "
        f"--out-plot {q(stress_plot)}"
    )
    run(
        f"{q(PYTHON_EXE)} scripts/hallucination_calibration.py "
        f"--csv {q(scenario_csv)} "
        f"--out {q(calibration_json)} "
        f"--out-by-ticker {q(calibration_by_ticker)}"
    )
    run(
        f"{q(PYTHON_EXE)} scripts/scenario_delta_report.py "
        f"--csv {q(scenario_csv)} "
        f"--out {q(scenario_delta_csv)}"
    )

    scenario_df = pd.read_csv(scenario_csv)
    required_meta = [
        "objective_track", "energy_component", "component_split_mode",
        "encoder_checkpoint", "critic_checkpoint", "train_neg_mode",
    ]
    missing = [c for c in required_meta if c not in scenario_df.columns]
    assert not missing, f"Scenario CSV missing metadata columns: {missing}"

    assert scenario_df["objective_track"].astype(str).str.lower().eq("critic").all(), "Scenario rows should be critic-tracked"
    assert scenario_df["energy_component"].astype(str).str.lower().eq("critic_energy").all(), "Scenario energy should be critic_energy"

    print("scenario rows:", len(scenario_df))
    meta_view = [c for c in [
        "scenario_id", "graph_index", "date", "target_ticker", "ticker", "series",
        "objective_track", "energy_component", "component_split_mode"
    ] if c in scenario_df.columns]
    print(scenario_df[meta_view].head(20).to_string(index=False))

    stress_df = pd.read_csv(stress_csv)
    print("stress rows:", len(stress_df))
    print(stress_df.head(20).to_string(index=False))

    with Path(calibration_json).open() as f:
        calib = json.load(f)
    print("calibration summary:")
    print({k: calib.get(k) for k in ["num_pairs", "num_points", "num_tickers", "corr_real_hall", "mae", "js_divergence", "tail_ratio_p99"]})

    if scenario_delta_csv.exists():
        delta_df = pd.read_csv(scenario_delta_csv)
        print("scenario_delta rows:", len(delta_df))
        if not delta_df.empty:
            print(delta_df.head(10).to_string(index=False))
else:
    print("Skipping scenario-dependent downstream steps: stress_test_report, hallucination_calibration, scenario_delta_report.")

if RUN_EXTRA_DIAGNOSTICS:
    if scenario_preflight_ok:
        run(
            f"{q(PYTHON_EXE)} scripts/feature_attribution.py "
            f"--config {q(runtime_config)} "
            f"--graphs {q(graphs_pt)} "
            f"--model {q(encoder_ckpt)} "
            f"--critic-model {q(critic_ckpt)} "
            f"--num-graphs {int(ATTR_NUM_GRAPHS)} "
            f"--out {q(feature_attr_csv)}"
        )
        run(
            f"{q(PYTHON_EXE)} scripts/embedding_diagnostics.py "
            f"--config {q(runtime_config)} "
            f"--graphs {q(graphs_pt)} "
            f"--model {q(encoder_ckpt)} "
            f"--method {q(EMBED_METHOD)} "
            f"--out {q(embedding_csv)}"
        )
    else:
        print("Skipping feature_attribution + embedding_diagnostics because scenario preflight failed.")
else:
    print("Skipping feature_attribution + embedding_diagnostics (RUN_EXTRA_DIAGNOSTICS=False)")

if feature_attr_csv.exists():
    fa_df = pd.read_csv(feature_attr_csv)
    print("feature attribution rows:", len(fa_df))
    print(fa_df.sort_values("importance_norm", ascending=False).head(10).to_string(index=False))

if embedding_csv.exists():
    emb_df = pd.read_csv(embedding_csv)
    print("embedding rows:", len(emb_df))
    print(emb_df[[c for c in ["graph_index", "date", "proj_x", "proj_y", "projection_method"] if c in emb_df.columns]].head(10).to_string(index=False))


## Step 6 (Optional): Goodness Backtest

In [ ]:
if RUN_OPTIONAL_BACKTEST:
    run(
        f"{q(PYTHON_EXE)} scripts/goodness_backtest.py "
        f"--config {q(runtime_config)} "
        f"--out-csv {q(goodness_csv)} "
        f"--out-quantiles {q(goodness_quantiles)} "
        f"--out-plot {q(goodness_plot)} "
        f"--out-events {q(goodness_events)} "
        f"--out-strategy {q(goodness_strategy)} "
        f"--out-timeline {q(goodness_timeline)}"
    )
else:
    print("Skipping goodness_backtest (RUN_OPTIONAL_BACKTEST=False)")

## Final Artifact Inventory

In [ ]:
artifacts = [
    runtime_config,
    graphs_pt,
    encoder_ckpt,
    critic_ckpt,
    model_ckpt,
    benchmark_csv,
    benchmark_baseline_csv,
    benchmark_walk_forward_csv,
    benchmark_plot,
    benchmark_bar,
    sweep_csv,
    sweep_summary,
    sweep_plot,
    sweep_pareto,
    optuna_json,
    scenario_csv,
    scenario_diag,
    scenario_delta_csv,
    stress_csv,
    stress_plot,
    calibration_json,
    calibration_by_ticker,
    feature_attr_csv,
    feature_attr_energy_csv,
    embedding_csv,
    goodness_csv,
    goodness_quantiles,
    goodness_plot,
    goodness_events,
    goodness_strategy,
    goodness_timeline,
]

rows = []
for p in artifacts:
    rows.append({
        "path": str(p),
        "exists": p.exists(),
        "bytes": p.stat().st_size if p.exists() and p.is_file() else None,
    })

artifact_df = pd.DataFrame(rows)
print(artifact_df.to_string(index=False))

missing = artifact_df.loc[~artifact_df["exists"], "path"].tolist()
if missing:
    print()
    print("Missing artifacts (may be expected if optional steps were disabled):")
    for p in missing:
        print("-", p)
else:
    print()
    print("All tracked artifacts exist.")
